In [ ]:
# ============================================================
# utils/gcs_utils.ipynb — GCS helpers. %run after config.
# ============================================================
%run utils/config.ipynb

!pip install gcsfs google-cloud-storage -q

from google.cloud import storage
import pandas as pd, os

_client = storage.Client()

def upload_df(df, layer, name, fmt='parquet'):
    """Upload DataFrame to GCS layer. fmt = 'parquet' or 'csv'"""
    local = f'/tmp/{name}.{fmt}'
    if fmt == 'parquet':
        df.to_parquet(local, index=False)
    else:
        df.to_csv(local, index=False)
    gcs_path = PATHS[layer] + f'{name}.{fmt}'
    blob_path = gcs_path.replace(f'gs://{BUCKET_NAME}/', '')
    _client.bucket(BUCKET_NAME).blob(blob_path).upload_from_filename(local)
    print(f'  ✅ {name} → {gcs_path}')

def read_df(layer, name, fmt='parquet'):
    """Read DataFrame from GCS layer."""
    path = PATHS[layer] + f'{name}.{fmt}'
    opts = {'token': 'google_default'}
    if fmt == 'parquet':
        return pd.read_parquet(path, storage_options=opts)
    return pd.read_csv(path, storage_options=opts)

print('✅ GCS utils loaded')
